# 04 — Train: ARIMA

Evaluate recursive forecasting across the fixed nonseasonal SARIMAX configurations on the shared validation folds. Select by mean `CV_SELECTION_METRIC`, refit the winner, and evaluate the sealed test once. ARIMA fits the hourly target history and scores the common cohort.


In [ ]:
%load_ext autoreload
%autoreload 2

## Explicit configuration grid

Search `p,q ∈ {0,1,2,3}` and `d ∈ {0,1}`. An intercept is searched only for `d=0`; a constant in a differenced model is a drift rather than a level, so `d=1` models have none. That gives 48 candidates. Each fit uses 200 optimizer iterations.

Stage-2 fills are restored to missing, consistent with the target eligibility rule. Each fit uses the full observed training history from the first observation through its cutoff; missing hours stay missing and the state-space likelihood skips them, so nothing is forward-filled or truncated. Rolling forecasts retain missing hours as state transitions. Finite nonconverged fits remain eligible and convergence is logged. Any fold fit or forecast failure marks the candidate `failed` and excludes it from selection. Selection uses the shared `src.model_selection.select_candidate`; exact metric ties prefer lower p+q, lower d, no intercept, then ascending p,q.

`N_WORKERS` sets how many candidates are evaluated in separate processes, with one numerical-library thread per worker and sequential folds within each candidate. Set it to `1` for sequential execution. Higher counts increase memory use. Results are reported in grid order; MLflow logging, winner selection, and the final refit/test remain in the main process.


In [ ]:
import json
from pathlib import Path
from time import time_ns
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from IPython.display import display
from mlflow.tracking import MlflowClient
from tqdm.auto import tqdm

from src.arima import (
    candidate_grid,
    evaluate_candidates,
    fit_arima,
    load_arima_model,
    log_candidate,
    observed_hourly_history,
    rolling_forecasts,
    candidate_table,
    latest_complete_segment,
    save_arima_model,
    select_candidate,
)
from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.metrics import metric_tables
from src.plots import (
    arima_correlation_figure,
    arima_search_figure,
    cv_error_boxplots_figure,
    predicted_vs_actual_figure,
    regime_aggregate_figure,
    regime_horizon_figure,
    test_error_boxplots_figure,
)
from src.regime_persistence import (
    regime_mlflow_metrics,
    regime_mlflow_params,
    sealed_test_regime_tables,
)
from src.training import prediction_preview

CANDIDATES = candidate_grid()
N_WORKERS = 10  # Set to 1 for sequential evaluation.
MAXITER = 200
ACF_PACF_LAGS = 48
PREDICTION_PREVIEW_ROWS = 5

PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
TRAIN_PATH = PROCESSED_DIR / "all_stations_train_features.parquet"
TEST_PATH = PROCESSED_DIR / "all_stations_test_features.parquet"
MODEL_PATH = Path("models") / f"arima_{TARGET_STATION_ID}.joblib"
MODEL_METADATA_PATH = MODEL_PATH.with_suffix(".json")
MLFLOW_EXPERIMENT_NAME = "arima"
NOTEBOOK_EXECUTION_UUID = str(uuid4())

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
MLFLOW_CLIENT = MlflowClient()

## Shared evaluation cohort and separate hourly history

`load_joined_dataset()` validates the configured station, horizon, predictors, targets, and provenance. We score exactly the same eligible issue times as the other training notebooks. ARIMA uses the unfiltered hourly history.

Training and test histories remain separate until the final rolling evaluation. The combined `target_context_series` remains plotting-only. All model timestamps are UTC.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    TRAIN_PATH,
    TEST_PATH,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
contract = dataset.contract
TARGET_COLUMNS = list(contract.target_columns)
train_rows = dataset.train_rows
test_rows = dataset.test_rows
train_history = observed_hourly_history(dataset.target_train_history)

COMMON_PARAMS = {
    **dataset.input_hashes,
    "station_id": TARGET_STATION_ID,
    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
    "candidate_count": len(CANDIDATES),
    "n_workers": N_WORKERS,
    "selection_metric": CV_SELECTION_METRIC,
    "maxiter": MAXITER,
    "training_data_policy": "full observed history through last training issue time + horizon; imputed as missing",
    "feature_count": 1,
    "feature_columns": json.dumps([f"{TARGET_STATION_ID}__water_level"]),
    "n_validation_folds": len(dataset.folds),
    "initial_train_fraction": INITIAL_TRAIN_FRACTION,
    "embargo_rows": EMBARGO_HOURS,
    "eligible_train_rows": len(train_rows),
    "eligible_test_rows": len(test_rows),
    "raw_train_rows": dataset.raw_row_counts["train"],
    "raw_test_rows": dataset.raw_row_counts["test"],
    "update_policy": "causal observed-level updates",
}
display(
    pd.DataFrame(
        [
            {
                "train_hours": len(train_history),
                "missing_observed_train_hours": int(train_history.isna().sum()),
                "eligible_train_origins": len(train_rows),
                "eligible_test_origins": len(test_rows),
            }
        ]
    )
)

## ACF and PACF — before model fitting

These plots use only the **latest gap-free stretch of the first fold's training history**, excluding all validation and test observations; ACF/PACF estimates need consecutive hours. Level and first-difference panels help distinguish persistence from short-term dependence; they do not manually set the differencing order. Lag counts adapt to the available segment length.

In [ ]:
first_train_indices, _ = dataset.folds[0]
first_fold_cutoff = train_rows.iloc[first_train_indices[-1]]["timestamp"] + pd.Timedelta(
    hours=FORECAST_HORIZON_HOURS
)
diagnostic_history = latest_complete_segment(train_history.loc[:first_fold_cutoff])
correlation_figure = arima_correlation_figure(diagnostic_history, lags=ACF_PACF_LAGS)
plt.show()

## Time-series cross-validation

Each fixed configuration fits the observed levels through the fold's final training issue time plus the 24-hour horizon: the latest level any training target uses, as for the other models. The shared 24-row embargo keeps this cutoff before the first validation issue time, so the expanding folds and embargo are unchanged. Each fitted state advances through elapsed hours. Parameters remain fixed.

One MLflow candidate parent and its fold children record configuration, fitted history, boundaries, convergence, failures, and aggregate/per-horizon scores. Selection requires every fold; execution stops if no candidate qualifies.


In [ ]:
evaluations = []
candidate_results = evaluate_candidates(
    CANDIDATES,
    train_history,
    train_rows,
    dataset.folds,
    contract=contract,
    n_workers=N_WORKERS,
    maxiter=MAXITER,
)
for candidate_number, evaluation in enumerate(
    tqdm(candidate_results, total=len(CANDIDATES), desc="ARIMA CV search", unit="candidate"),
    start=1,
):
    # Runs are logged after worker-side evaluation, so back-date their times.
    candidate_run = MLFLOW_CLIENT.create_run(
        experiment_id=MLFLOW_EXPERIMENT.experiment_id,
        start_time=evaluation.started_at_ms,
        run_name=f"arima_candidate_{candidate_number:03d}",
        tags={
            "phase": "cv",
            "run_type": "candidate_parent",
            "execution_uuid": NOTEBOOK_EXECUTION_UUID,
        },
    )
    with mlflow.start_run(run_id=candidate_run.info.run_id):
        mlflow.log_params(COMMON_PARAMS)
        log_candidate(evaluation)
    MLFLOW_CLIENT.set_terminated(
        candidate_run.info.run_id, end_time=evaluation.ended_at_ms
    )
    evaluations.append(evaluation)
    tqdm.write(f"Candidate {candidate_number}/{len(CANDIDATES)}: {evaluation.candidate}")

selected_order, selected_intercept = select_candidate(
    candidate_table(evaluations), CV_SELECTION_METRIC
)
winner = next(
    result
    for result in evaluations
    if (result.candidate.order, result.candidate.intercept)
    == (selected_order, selected_intercept)
)
cv_summary = winner.summary
cv_aggregate_metrics = winner.aggregate
cv_horizon_metrics = winner.horizons
comparison = pd.DataFrame(
    [
        {
            **result.candidate.parameters(),
            "status": "ok" if result.summary else "failed",
            **result.summary,
        }
        for result in evaluations
    ]
)
print(f"{comparison['status'].eq('failed').sum()} of {len(comparison)} candidates failed")
display(comparison)
display(pd.DataFrame(winner.fold_details))
search_figure = arima_search_figure(comparison, metric=CV_SELECTION_METRIC)
display(search_figure)
cv_figure = cv_error_boxplots_figure(
    cv_horizon_metrics, TARGET_COLUMNS, title="Selected ARIMA CV errors"
)
plt.show()

## Final training fit and one sealed-test evaluation

Refit the winning fixed configuration on training data. A failed refit stops execution; no replacement is selected. Fitting uses the full observed history through the final training issue time plus the 24-hour horizon, the same rule as in each fold. Test observations never select orders or change fitted parameters.

As test time advances, observations become available for state updates. Future observations are never used for an earlier origin. The model saved below is the **training-state snapshot**, unchanged by test evaluation, so it can be reloaded and advanced independently. Every diagnostic below reuses this single set of test predictions.

In [ ]:
sealed_test_started_at_ms = time_ns() // 1_000_000
final_progress = tqdm(total=2, desc="ARIMA final fit", unit="step")
final_model, final_fit_details = fit_arima(
    train_history,
    cutoff=train_rows["timestamp"].iloc[-1] + pd.Timedelta(hours=FORECAST_HORIZON_HOURS),
    contract=contract,
    candidate=winner.candidate,
    maxiter=MAXITER,
)
final_progress.update()
final_progress.set_description("ARIMA sealed-test forecast")

test_history = observed_hourly_history(dataset.target_test_history)
if train_history.index[-1] >= test_history.index[0]:
    raise ValueError("ARIMA requires strictly separated train/test histories")
forecast_history = pd.concat([train_history, test_history]).asfreq("h")
test_predictions = rolling_forecasts(
    final_model, forecast_history, test_rows["timestamp"]
)
final_progress.update()
final_progress.close()
display(pd.DataFrame([final_fit_details]))
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=TARGET_STATION_ID,
)
regime_definition, regime_aggregate_metrics, regime_horizon_metrics = (
    sealed_test_regime_tables(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        target_columns=TARGET_COLUMNS,
        station_id=TARGET_STATION_ID,
        quartile_cutoffs_cm=dataset.target_water_level_quartile_cutoffs_cm,
        quartile_reference_count=dataset.target_water_level_quartile_reference_count,
    )
)
sealed_test_ended_at_ms = time_ns() // 1_000_000
prediction_table = prediction_preview(
    test_rows, test_predictions, target_columns=TARGET_COLUMNS
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_table.head(PREDICTION_PREVIEW_ROWS))

## Record test results and save the model contract

The saved manifest describes only the model, hourly input/output contract, selected orders, state timestamp, missing-observation policy, artifact hash, and execution UUID. CV/test scores, convergence, training interval, and regime diagnostics remain in MLflow.

In [ ]:
sealed_run = MLFLOW_CLIENT.create_run(
    experiment_id=MLFLOW_EXPERIMENT.experiment_id,
    start_time=sealed_test_started_at_ms,
    run_name="arima_sealed_test",
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
)
with mlflow.start_run(run_id=sealed_run.info.run_id):
    mlflow.log_params(
        {
            **COMMON_PARAMS,
            **final_fit_details,
            **regime_mlflow_params(regime_definition),
            "scored_issue_times": len(test_rows),
            "test_start": test_rows["timestamp"].iloc[0].isoformat(),
            "test_end": test_rows["timestamp"].iloc[-1].isoformat(),
        }
    )
    mlflow.log_figure(search_figure, "candidate_search.png")
    mlflow.log_figure(cv_figure, "selected_cv_errors.png")
    mlflow.log_figure(correlation_figure, "training_acf_pacf.png")
    mlflow.log_table(comparison, "candidate_comparison.json")
    mlflow.log_metrics(
        {
            **{
                f"test_{metric}": float(aggregate_metrics.iloc[0][metric])
                for metric in ("mae", "rmse", "me", "r2")
            },
            **{
                f"test_{metric}_horizon_{int(row.horizon_hours):02d}": float(
                    getattr(row, metric)
                )
                for row in per_horizon_metrics.itertuples()
                for metric in ("mae", "rmse", "me", "r2")
            },
            **regime_mlflow_metrics(regime_aggregate_metrics, regime_horizon_metrics),
        }
    )
    figures = {
        "test_errors.png": test_error_boxplots_figure(
            test_rows,
            test_predictions,
            per_horizon_metrics,
            TARGET_COLUMNS,
            title="ARIMA sealed-test errors",
        ),
        "predicted_vs_actual.png": predicted_vs_actual_figure(
            test_rows[TARGET_COLUMNS],
            test_predictions,
            TARGET_COLUMNS,
            title="ARIMA predicted vs actual",
        ),
    }
    for filename, figure in figures.items():
        mlflow.log_figure(figure, filename)
        display(figure)
        plt.close(figure)
    save_arima_model(
        final_model,
        MODEL_PATH,
        MODEL_METADATA_PATH,
        execution_uuid=NOTEBOOK_EXECUTION_UUID,
    )
    mlflow.log_artifact(str(MODEL_PATH), artifact_path="model")
    mlflow.log_artifact(str(MODEL_METADATA_PATH), artifact_path="model")
MLFLOW_CLIENT.set_terminated(sealed_run.info.run_id, end_time=sealed_test_ended_at_ms)
print(f"Saved {MODEL_PATH} and {MODEL_METADATA_PATH}")

# ARIMA saved-model review

Start here in a fresh kernel. This section reads the validated saved model and its exact MLflow execution; it does not fit candidates.


In [ ]:
import json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from mlflow.tracking import MlflowClient
from threadpoolctl import threadpool_limits

from src.arima import load_arima_model, rolling_forecasts
from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WATER_LEVEL_ALARM_THRESHOLD_CM,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.evaluation import load_saved_model_review, verify_saved_forecast_metrics
from src.metrics import metric_tables
from src.plots import (
    forecast_window_figures,
    regime_aggregate_figure,
    regime_horizon_figure,
)
from src.regime_persistence import sealed_test_regime_tables

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
review_dir = Path("data/processed/joined")
review_model_path = Path("models") / f"arima_{TARGET_STATION_ID}.joblib"
review_manifest_path = review_model_path.with_suffix(".json")
review_dataset = load_joined_dataset(
    review_dir / "all_stations_feature_metadata.json",
    review_dir / "all_stations_train_features.parquet",
    review_dir / "all_stations_test_features.parquet",
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
review_contract = review_dataset.contract
review_model = load_arima_model(
    review_model_path, review_manifest_path, contract=review_contract
)
review_manifest = json.loads(review_manifest_path.read_text())
review_uuid = review_manifest.get("execution_uuid")
if not isinstance(review_uuid, str) or not review_uuid:
    raise ValueError("Saved ARIMA manifest has no execution_uuid")
review_targets = list(review_contract.target_columns)
from src.arima import observed_hourly_history
from src.plots import arima_search_figure

review_parameters = {
    "order": str(review_model.order),
    "intercept": str(review_model.intercept),
}
review = load_saved_model_review(
    "arima", review_uuid,
    input_hashes=review_dataset.input_hashes,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    scored_issue_times=len(review_dataset.test_rows),
    selected_parameters=review_parameters,
    quartile_cutoffs_cm=review_dataset.target_water_level_quartile_cutoffs_cm,
    quartile_reference_count=review_dataset.target_water_level_quartile_reference_count,
    alarm_threshold_cm=WATER_LEVEL_ALARM_THRESHOLD_CM,
    client=MlflowClient(),
)
print(f"Reviewing saved ARIMA execution {review.execution_uuid}")


## Recorded CV candidates and selected test result

Failed candidates remain visible in the table and are excluded from performance plots.


In [ ]:
review_candidates = review.candidates.copy()
candidate_columns = ["status", "selected", *["order", "intercept"],
                     "cv_mae_mean", "cv_mae_std", "cv_rmse_mean", "cv_rmse_std"]
display(review_candidates[candidate_columns])
selected_summary = {"execution_uuid": review.execution_uuid,
                    **{f"cv_{metric}_mean": review.selected_candidate[f"cv_{metric}_mean"]
                       for metric in ("mae", "rmse", "me", "r2")},
                    **{f"test_{metric}": review.test_aggregate.iloc[0][metric]
                       for metric in ("mae", "rmse", "me", "r2")}}
display(pd.DataFrame([selected_summary]))
display(review.cv_horizons)
display(review.test_aggregate)
display(review.test_horizons)

successful = review_candidates.loc[review_candidates["status"].eq("ok")].copy()
print(f"{review_candidates['status'].eq('failed').sum()} of {len(review_candidates)} candidates failed")
display(arima_search_figure(review_candidates, metric=CV_SELECTION_METRIC))
for parameter in ("order", "intercept"):
    figure = go.Figure()
    for value, group in successful.groupby(parameter, sort=True):
        figure.add_trace(go.Box(
            x=[str(value)] * len(group), y=group[f"cv_{CV_SELECTION_METRIC}_mean"],
            name=str(value), boxpoints="all",
        ))
    figure.update_layout(title=f"ARIMA CV {CV_SELECTION_METRIC.upper()} by {parameter}",
                         xaxis_title=parameter, yaxis_title="CV mean error (cm)")
    display(figure)


## Score the saved state on the eligible sealed test cohort

The original hourly update policy is used, including explicit missing hours. Recomputed metrics must agree with the linked MLflow run.


In [ ]:
review_test_rows = review_dataset.test_rows
review_train_history = observed_hourly_history(review_dataset.target_train_history)
review_test_history = observed_hourly_history(review_dataset.target_test_history)
if review_train_history.index[-1] >= review_test_history.index[0]:
    raise ValueError("ARIMA train and test histories overlap")
review_history = pd.concat([review_train_history, review_test_history]).asfreq("h")
review_predictions = rolling_forecasts(
    review_model, review_history, review_test_rows["timestamp"]
)
review_aggregate, review_horizons = metric_tables(
    review_test_rows[review_targets], review_predictions,
    target_columns=review_targets, station_id=TARGET_STATION_ID,
)
_, review_regime_aggregate, review_regime_horizons = sealed_test_regime_tables(
    review_test_rows[review_targets], review_predictions,
    target_columns=review_targets, station_id=TARGET_STATION_ID,
    quartile_cutoffs_cm=review_dataset.target_water_level_quartile_cutoffs_cm,
    quartile_reference_count=review_dataset.target_water_level_quartile_reference_count,
)
verify_saved_forecast_metrics(
    review, review_aggregate, review_horizons,
    review_regime_aggregate, review_regime_horizons,
)
print("Saved-model aggregate, horizon, quartile, and alarm scores match MLflow")
display(review_aggregate)
display(review_horizons)


## Predictions across horizons and complete forecast windows


In [ ]:
prediction_columns = [f"prediction_{target}" for target in review_targets]
prediction_table = review_test_rows[["timestamp", *review_targets]].rename(
    columns={"timestamp": "issue_time"}
).copy()
for column, values in zip(prediction_columns, review_predictions.T, strict=True):
    prediction_table[column] = values
horizons = list(range(1, len(review_targets) + 1))
labels = [f"H+{horizon:02d}" for horizon in horizons]
issue_times = pd.to_datetime(prediction_table["issue_time"], utc=True)
frames = []
for horizon, target, prediction in zip(horizons, review_targets, prediction_columns, strict=True):
    valid_times = issue_times + pd.to_timedelta(horizon, unit="h")
    frames.append(go.Frame(
        name=labels[horizon - 1],
        data=[
            go.Scattergl(x=valid_times, y=prediction_table[target], mode="lines", name="Actual"),
            go.Scattergl(x=valid_times, y=prediction_table[prediction], mode="lines", name="Prediction"),
        ],
    ))
time_series = go.Figure(
    data=frames[0].data, frames=frames,
    layout={
        "title": "Saved ARIMA predictions across forecast horizons",
        "xaxis_title": "Valid time", "yaxis_title": "Water level (cm)",
        "sliders": [{"active": 0, "currentvalue": {"prefix": "Forecast horizon: "},
                     "steps": [{"label": label, "method": "animate",
                                "args": [[label], {"mode": "immediate"}]} for label in labels]}],
    },
)
display(time_series)
windows = forecast_window_figures(
    prediction_table, review_dataset.target_context_series,
    water_level_column=f"{TARGET_STATION_ID}__water_level",
    imputed_column=f"{TARGET_STATION_ID}__imputed",
    prediction_columns=prediction_columns, target_columns=review_targets,
    horizons=horizons, label_prefix="ARIMA",
)
display(windows["best"])
display(windows["worst"])


## Absolute and signed error distributions


In [ ]:
signed_errors = review_predictions - review_test_rows[review_targets].to_numpy(dtype=float)
absolute_errors = np.abs(signed_errors)
absolute_figure = go.Figure(data=[
    go.Box(x=labels * len(absolute_errors), y=absolute_errors.reshape(-1), name="Absolute error"),
    go.Scatter(x=labels, y=review_horizons["mae"], mode="markers", name="MAE"),
    go.Scatter(x=labels, y=review_horizons["rmse"], mode="markers", name="RMSE"),
])
absolute_figure.update_layout(title="Saved ARIMA absolute errors", xaxis_title="Horizon", yaxis_title="cm")
display(absolute_figure)
signed_figure = go.Figure(data=[
    go.Box(x=labels * len(signed_errors), y=signed_errors.reshape(-1), name="Signed error"),
    go.Scatter(x=labels, y=signed_errors.mean(axis=0), mode="markers", name="ME"),
])
signed_figure.add_hline(y=0, line_dash="dash")
signed_figure.update_layout(title="Saved ARIMA signed errors", xaxis_title="Horizon", yaxis_title="Prediction minus actual (cm)")
display(signed_figure)


## ARIMA model and fit diagnostics


In [ ]:
display(review_model.results.summary())
display(pd.DataFrame([{
    "order": review_model.order, "seasonal_order": review_model.seasonal_order,
    "intercept": review_model.intercept,
    "state_time": review_model.state_time,
    "aic": review_model.results.aic, "aicc": review_model.results.aicc,
    "bic": review_model.results.bic,
}]))


## Quartile and alarm diagnostics


In [ ]:
display(review.regime_aggregate)
display(review.regime_horizons)
for regime_group in ("quartile", "alarm"):
    display(regime_aggregate_figure(
        review.regime_aggregate, regime_group=regime_group, model_label="ARIMA"
    ))
    for metric in ("mae", "rmse", "me"):
        display(regime_horizon_figure(
            review.regime_horizons, regime_group=regime_group,
            metric=metric, model_label="ARIMA",
        ))
